# Heatmap Generation — SQ-MIL

Generate per-subtype spatial prediction heatmaps from a trained Stage 2 checkpoint.

**Pipeline**: Load checkpoint → run inference on test split → produce per-patch CSV → overlay predictions on WSI thumbnail.

Color scheme: CC (Red), EC (Green), HGSC (Yellow), LGSC (Blue), MC (Purple)

In [ ]:
# ============================================================
# CONFIGURATION — edit these paths before running
# ============================================================

# --- Checkpoint ---
CHECKPOINT_PATH = 'results/stage2/fold0/best_model.pth'
CONFIG_PATH     = 'configs/ovarian_conch_s2.yaml'
FOLD            = 0
STAGE           = 2     # 2 = Stage 2 (with refinement); 1 = Stage 1 only

# --- Data paths ---
WSI_DIR         = 'data/wsi'          # directory containing .tif WSIs
DATA_ROOT       = 'data'              # parent of embeddings/, superpixels/, splits/, labels.csv
EMBEDDING_DIR   = None                # override: set to skip DATA_ROOT/embeddings
SUPERPIXEL_DIR  = None                # override: set to skip DATA_ROOT/superpixels

# --- Output ---
OUTPUT_DIR      = 'results/heatmaps'

# --- Heatmap appearance ---
THUMBNAIL_SIZE  = 2048
OVERLAY_ALPHA   = 0.50
GAUSSIAN_SIGMA  = 5.0

# --- Display ---
N_DISPLAY       = 6     # number of heatmaps to show inline

In [ ]:
# ============================================================
# SETUP — imports, config loading, path resolution
# ============================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if (PROJECT_ROOT / 'src').is_dir():
    pass
elif (PROJECT_ROOT.parent / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError('Cannot find project root (expected src/ directory)')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
import torch
from torch.utils.data import DataLoader

from src.training.trainer import SMMILeTrainer
from src.datasets.mil_dataset import build_dataset
from src.visualization.heatmap import HeatmapGenerator

# Load and patch config
with open(PROJECT_ROOT / CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

paths = cfg.setdefault('paths', {})
paths['data_root']      = str(PROJECT_ROOT / DATA_ROOT)
paths['embedding_dir']  = str(PROJECT_ROOT / (EMBEDDING_DIR  or Path(DATA_ROOT) / 'embeddings'))
paths['superpixel_dir'] = str(PROJECT_ROOT / (SUPERPIXEL_DIR or Path(DATA_ROOT) / 'superpixels'))
paths['split_dir']      = str(PROJECT_ROOT / DATA_ROOT / 'splits')
paths['labels_csv']     = str(PROJECT_ROOT / DATA_ROOT / 'labels.csv')
paths['output_dir']     = str(PROJECT_ROOT / OUTPUT_DIR)

ckpt_path = PROJECT_ROOT / CHECKPOINT_PATH
wsi_dir   = PROJECT_ROOT / WSI_DIR
out_dir   = PROJECT_ROOT / OUTPUT_DIR

print(f'Project root : {PROJECT_ROOT}')
print(f'Checkpoint   : {ckpt_path}')
print(f'WSI dir      : {wsi_dir}')
print(f'Embeddings   : {paths["embedding_dir"]}')
print(f'Superpixels  : {paths["superpixel_dir"]}')
print(f'Output       : {out_dir}')
print(f'Device       : {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# ============================================================
# INFERENCE — load checkpoint, run on test split, save CSV
# ============================================================

# Build test DataLoader
split_csv  = Path(paths['split_dir']) / f'splits_{FOLD}.csv'
labels_csv = paths['labels_csv']
test_ds    = build_dataset(
    split_csv      = split_csv,
    labels_csv     = labels_csv,
    embedding_dir  = paths['embedding_dir'],
    superpixel_dir = paths['superpixel_dir'],
    split          = 'test',
    augment        = False,
)
test_loader = DataLoader(
    test_ds,
    batch_size  = 1,
    shuffle     = False,
    num_workers = cfg.get('hardware', {}).get('num_workers', 4),
    pin_memory  = torch.cuda.is_available(),
)
print(f'Test slides: {len(test_ds)}')

# Load model and evaluate
trainer = SMMILeTrainer(model=None, config=cfg, fold_idx=FOLD)
trainer.load_checkpoint(ckpt_path, stage=STAGE)

metrics = trainer.evaluate(test_loader)

inst_csv_dir = out_dir.parent / OUTPUT_DIR.split('/')[0] / f'stage{STAGE}' / f'fold{FOLD}'
# The trainer writes inst_predictions to its own output_dir/foldN/
inst_csv_dir = Path(paths['output_dir']) / f'fold{FOLD}'

print(f'\nMetrics:')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:20s}: {v:.4f}')
print(f'\nInstance predictions saved to: {inst_csv_dir}')

In [ ]:
# ============================================================
# HEATMAP GENERATION
# ============================================================

gen = HeatmapGenerator(
    thumbnail_size = (THUMBNAIL_SIZE, THUMBNAIL_SIZE),
    overlay_alpha  = OVERLAY_ALPHA,
    gaussian_sigma = GAUSSIAN_SIGMA,
)

results = gen.generate_batch(
    wsi_dir         = str(wsi_dir),
    predictions_dir = str(inst_csv_dir),
    output_dir      = str(out_dir),
)

ok    = sum(1 for v in results.values() if not isinstance(v, Exception))
errs  = {k: v for k, v in results.items() if isinstance(v, Exception)}
print(f'\nGenerated: {ok} heatmaps')
if errs:
    print(f'Failed: {len(errs)}')
    for sid, exc in list(errs.items())[:5]:
        print(f'  {sid}: {exc}')

In [ ]:
# ============================================================
# DISPLAY HEATMAPS INLINE
# ============================================================
import glob as _glob
from IPython.display import Image, display

heatmap_files = sorted(_glob.glob(str(out_dir / '*_heatmap.png')))
print(f'Total heatmap files: {len(heatmap_files)}')

for hf in heatmap_files[:N_DISPLAY]:
    print(Path(hf).name)
    display(Image(hf, width=900))

In [ ]:
# ============================================================
# (OPTIONAL) SINGLE-SLIDE DEEP DIVE
# Uncomment and set SLIDE_ID to regenerate one heatmap with
# custom settings.
# ============================================================

# import pandas as pd
# from IPython.display import Image, display
#
# SLIDE_ID = 'your_slide_id_here'
#
# csv_path = inst_csv_dir / f'inst_predictions_fold{FOLD}.csv'
# df = pd.read_csv(csv_path)
# slide_df = df[df['slide_id'] == SLIDE_ID].reset_index(drop=True)
# print(f'Patches for {SLIDE_ID}: {len(slide_df)}')
#
# gen_custom = HeatmapGenerator(
#     thumbnail_size = (4096, 4096),
#     overlay_alpha  = 0.60,
#     gaussian_sigma = 3.0,
# )
# out_path = out_dir / f'{SLIDE_ID}_heatmap_custom.png'
# gen_custom.generate(
#     wsi_path       = wsi_dir / f'{SLIDE_ID}.tif',
#     predictions_df = slide_df,
#     output_path    = out_path,
# )
# display(Image(str(out_path), width=1200))